# Generador 3D Multi-Vista ULTRA-HD (Hunyuan3D-2mv @ Octree 384, Rostro Natural Limpio) -> STL (Peana 30x30 mm)
Esta versión genera la malla nativa de alta densidad (`octree_resolution=384`, `steps=50`, `guidance_scale=5.0`) sin aplicar desplazamiento 2D ni filtros agresivos sobre el rostro, conservando las facciones 3D suaves y naturales y eliminando micro-partículas sueltas.

In [ ]:
!pip install -q gradio_client trimesh numpy scipy pillow
!git clone https://github.com/inigoku/mu.git /content/mu || (cd /content/mu && git pull)

In [ ]:
from pathlib import Path
import numpy as np
import trimesh
from gradio_client import Client, handle_file
from google.colab import files

front_png = Path('/content/mu/elektra_frontal.png')
back_png = Path('/content/mu/elektra_trasera.png')

print('Conectando con tencent/Hunyuan3D-2mv (octree=384, steps=50, guidance=5.0)...')
client = Client('tencent/Hunyuan3D-2mv')
result = client.predict(
    image=handle_file(str(front_png)),
    mv_image_front=handle_file(str(front_png)),
    mv_image_back=handle_file(str(back_png)),
    mv_image_left=None,
    mv_image_right=None,
    steps=50,
    guidance_scale=5.0,
    seed=1234,
    octree_resolution=384,
    check_box_rembg=False,
    num_chunks=20000,
    randomize_seed=False,
    api_name='/shape_generation'
)

mesh_path = result[0] if isinstance(result, (list, tuple)) else result
if isinstance(mesh_path, dict) and 'value' in mesh_path:
    mesh_path = mesh_path['value']

mesh = trimesh.load(mesh_path, force='mesh')
mesh.merge_vertices()

# Conservar únicamente el sólido principal (elimina partículas flotantes de Marching Cubes)
parts = mesh.split(only_watertight=False)
kept = [p for p in parts if len(p.faces) > 500]
mesh = trimesh.util.concatenate(kept)

v = mesh.vertices.copy()
if mesh.extents[1] > mesh.extents[2]:
    mesh.vertices = np.column_stack([v[:, 0], -v[:, 2], v[:, 1]])

# Escalar peana a 30.0 x 30.0 mm
mesh.vertices[:, 2] -= mesh.bounds[0, 2]
total_h = mesh.extents[2]
base_verts = mesh.vertices[mesh.vertices[:, 2] <= 0.12 * total_h]
base_w = float(base_verts[:, 0].max() - base_verts[:, 0].min())
base_d = float(base_verts[:, 1].max() - base_verts[:, 1].min())

sx = 30.0 / max(base_w, 1e-5)
sy = 30.0 / max(base_d, 1e-5)
sz = sx
cx = 0.5 * float(base_verts[:, 0].max() + base_verts[:, 0].min())
cy = 0.5 * float(base_verts[:, 1].max() + base_verts[:, 1].min())

mesh.vertices[:, 0] = (mesh.vertices[:, 0] - cx) * sx
mesh.vertices[:, 1] = (mesh.vertices[:, 1] - cy) * sy
mesh.vertices[:, 2] *= sz

output_stl = '/content/elektra_hunyuan3d_ULTRA_HD_30x30.stl'
mesh.export(output_stl)
print(f'STL limpio generado: {output_stl} | Triángulos: {len(mesh.faces):,} | Watertight: {mesh.is_watertight}')
files.download(output_stl)